# 🌍 Global Temperature Anomaly: Data Cleaning & Visualization

This notebook is part of the **Climate Data Hub**, a storytelling project by NewArts Toronto's Climate Generation Initiative. In this notebook, we process and visualize historical temperature anomaly data to support public awareness of climate change and global warming trends.

---

## 📁 Dataset

We are using the **NASA GISTEMP Land-Ocean Temperature Index**:
- Source: [NASA Goddard Institute for Space Studies (GISS)](https://data.giss.nasa.gov/gistemp/)
- Data fields include:
  - Year
  - Global Temperature Anomaly (°C) — unsmoothed
  - Loess-smoothed anomaly
  - Optional: Zonal temperature regions (e.g., Arctic, Tropics)

---

## 🔧 1. Data Cleaning

Steps:
- Load raw `.csv` or `.txt` file
- Rename columns and convert data types
- Filter relevant years (e.g., 1880–present)
- Handle missing or anomalous values (if any)
- Prepare data for visualization (melt, reshape, etc.)

---

## 📊 2. Data Visualization

We will create four core visualizations for this section:

### 2.1 Long-Term Global Warming Trend
- Line chart of global temperature anomalies (1880–present)
- Includes raw and smoothed values
- Annotated thresholds at +1.0°C and +1.5°C (Paris Agreement targets)

### 2.2 Decade-by-Decade Change
- Bar or line plot showing average anomaly per decade
- Highlights acceleration of warming over time

### 2.3 Regional Warming Disparities
- Bar or heatmap comparing warming in different zones (e.g., Arctic vs. Tropics)
- Requires GISTEMP zonal dataset

### 2.4 Threshold Breach Timeline
- Annotated line chart showing when global temps passed key thresholds:
  - 0.5°C, 1.0°C, and approaching 1.5°C

---

## 🧠 Notes

- All charts will be exported and saved as `.html` using Plotly for web embedding.
- Final visualizations will be integrated into the [Climate Data Hub website](https://www.newartsto.org/global-warming).



In [131]:
# 📦 Import standard libraries
import pandas as pd
import numpy as np
from io import StringIO
import nb_black
import requests
import os
from skmisc.loess import loess
from scipy.stats import zscore
from prophet import Prophet
import plotly.graph_objs as go
import plotly.express as px

# 📊 For quick visual checks
import matplotlib.pyplot as plt
import seaborn as sns


# 🔧 Display options for better readability
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [89]:
# load in .txt file
df = pd.read_csv("../data/graph.txt", delim_whitespace=True, header=None)

# set columns
df.columns = ['Year', 'No_Smoothing', 'Lowess_Smoothing']

# Organize data by decade
df["Decade"] = df["Year"] // 10 * 10

# Calculating between-decade means
df_decade = df.groupby("Decade")["No_Smoothing"].mean().reset_index().rename(columns={"No_Smoothing": "Mean_Anomaly"})

# Convert Years column into string, and add s to the end of it
df_decade["Decade"] = df_decade["Decade"].astype(str) + "s"

# Calculating absolute difference between decade means
df_decade["Abs_Change"] = df_decade["Mean_Anomaly"].diff()
df_decade["Abs_Change"].fillna(0, inplace=True)


In [192]:
def plot_decadal_temperature_anomaly(df_decade):
    fig = go.Figure()

    base_font = dict(family="Arial, sans-serif", size=14, color="white")
    title_font = dict(size=24, family="Arial, sans-serif")
    title_text = "📊 <b>Average Global Temperature Anomaly by Decade</b>"
    annotation_style = dict(size=12, color="lightgrey")

    colorscale = px.colors.sequential.Oranges

    fig.add_trace(go.Bar(
        x=df_decade["Decade"],
        y=df_decade["Mean_Anomaly"],
        marker=dict(
            color=df_decade["Mean_Anomaly"],
            colorscale=colorscale,
            colorbar=dict(title="Anomaly (°C)", x=1.02, len=1)
        ),
        hovertemplate="Decade: %{x}<br>Anomaly: %{y:.2f}°C<extra></extra>",
        name="Mean Temperature",
    ))

    subtitle = (
        "<b>Decadal averages show a clear and accelerating warming trend since the late 19th century.</b><br>"
        "<b>This highlights the long-term impact of climate change. "
        "Source: <a href='https://data.giss.nasa.gov/gistemp/graphs/graph_data/"
        "Global_Mean_Estimates_based_on_Land_and_Ocean_Data/graph.txt' "
        "style='color:lightblue;'>NASA GISS</a></b>"
    )

    fig.add_annotation(
        text=subtitle,
        xref="paper",
        yref="paper",
        x=0,
        y=1.13,
        showarrow=False,
        font=annotation_style,
        align="left",
        borderpad=4,
        bordercolor="rgba(0,0,0,0)",
        bgcolor="rgba(0,0,0,0)"
    )

    fig.add_hline(
        y=0,
        line=dict(color="lightgrey", dash="dot"),
        annotation_text="Baseline (0°C)",
        annotation_position="top left",
        annotation_yshift=10,
        opacity=0.7,
    )

    fig.update_layout(
        title=dict(text=title_text, font=dict(size=28)),
        xaxis_title="Decades",
        yaxis_title="Temperature Anomaly (°C)",
        template="plotly_dark",
        hovermode="x unified",
        font=base_font,
        title_font=title_font,
        autosize=True,
        margin=dict(l=40, r=40, t=120, b=40),
    )

    return fig

In [18]:
"""
This module adds reference bands and climate thresholds to a Plotly figure.

It provides a utility function to enhance temperature anomaly visualizations
with shaded warming zones and climate threshold lines.
"""


def add_reference_bands(fig, x_start, x_end):
    """Add shaded bands and threshold lines for climate warming levels.

    Modifies a Plotly figure in-place by adding:
    - A light orange band (1.0°C to 1.5°C) for warning
    - A light red band (1.5°C to 2.0°C) for danger
    - A baseline line at 0.0°C
    - A dashed orange line at 1.0°C
    - A dashed red line at 1.5°C

    Parameters
    ----------
    fig : plotly.graph_objects.Figure
        The Plotly figure to update.
    x_start : int or float
        Start of the x-axis range (e.g., earliest year).
    x_end : int or float
        End of the x-axis range (e.g., latest year).

    Returns
    -------
    None
        The figure is modified directly.
    """
    # Add warning band (1.0°C to 1.5°C)
    fig.add_shape(
        type="rect",
        x0=x_start,
        x1=x_end,
        y0=1,
        y1=1.5,
        fillcolor="rgba(255, 165, 0, 0.1)",
        layer="below",
        line_width=0,
    )

    # Add danger band (above 1.5°C)
    fig.add_shape(
        type="rect",
        x0=x_start,
        x1=x_end,
        y0=1.5,
        y1=2,
        fillcolor="rgba(255, 99, 71, 0.1)",
        layer="below",
        line_width=0,
    )

    # Add baseline reference line (0°C)
    fig.add_hline(
        y=0.0,
        annotation_text="<b>Baseline (0°C)</b>",
        annotation_position="top left",
    )

    # Add 1.0°C warning threshold line
    fig.add_hline(
        y=1.0,
        line=dict(color="orange", dash="dot"),
        annotation_text="<b>1.0°C Threshold</b>",
        annotation_position="top left",
    )

    # Add 1.5°C danger threshold line
    fig.add_hline(
        y=1.5,
        line=dict(color="darkred", dash="dot"),
        annotation_text="<b>1.5°C Threshold</b>",
        annotation_position="top left",
    )

    return fig


In [195]:
def plot_decadal_temperature_anomaly(df_decade):
    """Plot average global temperature anomaly by decade using Plotly.

    Parameters
    ----------
    df_decade : pandas.DataFrame
        A DataFrame containing:
        - 'Decade': int
        - 'Mean_Anomaly': float

    Returns
    -------
    plotly.graph_objects.Figure
        A styled Plotly figure with colored bars and embedded subtitle.
    """
    # Fonts and styles
    base_font = dict(family="Arial, sans-serif", size=14, color="white")
    title_font = dict(size=24, family="Arial, sans-serif")

    # Combine title + subtitle into one HTML string
    title_text = (
        "📊 <b>Average Global Temperature Anomaly by Decade</b><br>"
        "<span style='font-size:14px; color:lightgrey;'>"
        "Decadal averages show an accelerating warming trend since the late 19th century.<br>"
        "This highlights the long-term impact of climate change. "
        "Source: <a href='https://data.giss.nasa.gov/gistemp/graphs/graph_data/"
        "Global_Mean_Estimates_based_on_Land_and_Ocean_Data/graph.txt' style='color:lightblue;'>NASA GISS</a>"
        "</span>"
    )

    # Create figure
    fig = go.Figure()

    # Add bars
    fig.add_trace(go.Bar(
        x=df_decade["Decade"],
        y=df_decade["Mean_Anomaly"],
        marker=dict(
            color=df_decade["Mean_Anomaly"],
            colorscale=px.colors.sequential.Oranges,
            colorbar=dict(title="Anomaly (°C)", x=1.02, len=1),
        ),
        hovertemplate="Decade: %{x}<br>Anomaly: %{y:.2f}°C<extra></extra>",
        name="Mean Temperature"
    ))

    # Add 0°C baseline
    fig.add_hline(
        y=0,
        line=dict(color="lightgrey", dash="dot"),
        annotation_text="Baseline (0°C)",
        annotation_position="top left",
        annotation_yshift=10,
        opacity=0.7
    )

    # Final layout
    fig.update_layout(
        title=dict(text=title_text, font=title_font, x=0.01, xanchor="left"),
        xaxis_title="Decades",
        yaxis_title="Temperature Anomaly (°C)",
        template="plotly_dark",
        hovermode="x unified",
        font=base_font,
        autosize=True,
        margin=dict(l=40, r=40, t=130, b=40),  # Enough space for multiline title
    )

    return fig

In [196]:
fig = plot_temperature_trend(df)
fig_with_bands = add_reference_bands(
    plot_temperature_trend(df), df["Year"].min(), df["Year"].max())
fig_decade = plot_decadal_mean_anomaly(df_decade)
fig.show()
fig_with_bands.show()
fig_decade.show()

In [197]:
# export graph with all interactive elements
fig.write_html("../dataviz/python/annual-temperature-anomaly/index.html")
fig_with_bands.write_html(
    "../dataviz/python/annual-temperature-anomaly-bands/index.html"
)
fig_decade.write_html("../dataviz/python/decadal-mean-anomaly/index.html")